# Introduction to the Register Map (VCK190)

It is possible to access the register map for IP in the overlay directly. This functionality requires that the `.hwh` file is distributed along with the PDI. This can be generated from Vivado.

In this example we'll be looking at the AXI GPIO IP exposed by the VCK190 base overlay:

* `axi_gpio_led`      -- the 4 board LEDs (output)
* `axi_gpio_pb`        -- the 2 push-buttons (input)
* `axi_gpio_dip_sw`   -- the 4 DIP switches (input)

The convenience aliases `base.leds` / `base.buttons` / `base.switches` on the `BaseOverlay` wrap the default channel of each respective AXI GPIO.

In [ ]:
from pynq.overlays.base import BaseOverlay

base = BaseOverlay('base.pdi')

Grab the full driver objects for the GPIO IPs -- these are the ones the register map API is attached to (the convenience aliases on `base` are just single-channel views).

In [ ]:
btns   = base.axi_gpio_pb
leds   = base.axi_gpio_led
switches = base.axi_gpio_dip_sw

We can now print the register map and the current values of the registers by printing the representation of the `.register_map`.

In [ ]:
btns.register_map

To access values programmatically we can walk the object model. `GPIO_DATA.CH1_DATA` reflects the current live state of the input pins.

In [ ]:
btns.register_map.GPIO_DATA.CH1_DATA

If there is any more information on a register available it can be accessed using `help`. This will list any description of the register available as well as the subfields. Here we see that the `IP_IER` register is the interrupt enable register.

In [ ]:
help(btns.register_map.IP_IER)

Writing can be performed either to registers or fields. Note that register slice access uses RTL conventions (hi:lo inclusive) to match datasheet notation. These writes drive the 4 board LEDs via the AXI GPIO.

In [ ]:
leds.register_map.GPIO_DATA.CH1_DATA = 8
leds.register_map.GPIO_DATA[2:0] = 5

All four LEDs should now reflect `0b1101 = 0xd` (bit 3 from the scalar write, bits 2-0 from the slice write). Read back via the switches/buttons GPIOs doesn't apply here (they're input-only), but you can read the switch state:

In [ ]:
switches.register_map.GPIO_DATA.CH1_DATA

At present this functionality is limited to scalar registers in IP that has the attached metadata. This includes most IP in the Xilinx IP catalog and any HLS-generated IP that uses AXI-lite for control registers.

The same API also works on the AXI DMA (`base.axi_dma_0.register_map`), the AXI BRAM controller (`base.axi_bram_ctrl_0.register_map`), and the AXI UART Lite (`base.axi_uartlite_0.register_map`) -- try them if you're curious about the low-level view.